In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# 1. 确定设备
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# 2. 数据转换格式
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_data = datasets.FashionMNIST(root="data", train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST(root="data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64)

print(f"数据准备就绪！训练集：{len(train_data)}张，测试集：{len(test_data)}张")

数据准备就绪！训练集：60000张，测试集：10000张


The Kernel crashed while executing code in the current cell or a previous cell. <br>
Please review the code in the cell(s) to identify a possible cause of the failure. <br>
Click here for more info. <br>
View Jupyter log for further details.<br>
我也没搞清什么原因：我用uv pip install --upgrade torch torchvision torchaudio在终端执行一下，关闭重启就好了<br>

In [3]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device) 

        pred = model(X)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * len(X)
            print(f"Loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    model.eval() # 评估模式
    with torch.no_grad(): 
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [ ]:
epochs = 5 
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_loader, model, loss_fn, optimizer)
    test_loop(test_loader, model, loss_fn)
print("训练全部完成！")

Epoch 1<br>
Test Error: <br>
 Accuracy: 84.2%, Avg loss: 0.426461 <br>
<br>
Epoch 2<br>
Test Error: <br>
 Accuracy: 86.0%, Avg loss: 0.378429 <br>
<br>
Epoch 3<br>
Test Error: <br>
 Accuracy: 87.4%, Avg loss: 0.350893 <br>
<br>
Epoch 4<br>
Test Error: <br>
 Accuracy: 86.9%, Avg loss: 0.359255 <br>
<br>
Epoch 5<br>
Test Error: <br>
 Accuracy: 87.6%, Avg loss: 0.345491 <br>

In [ ]:
import matplotlib.pyplot as plt

classes = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", 
           "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

model.eval()
x, y = test_data[0][0], test_data[0][1] 
with torch.no_grad():
    x = x.to(device).unsqueeze(0)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'预测结果: "{predicted}", 实际结果: "{actual}"')

plt.imshow(test_data[0][0].squeeze(), cmap="gray")
plt.title(f"Predict: {predicted}")
plt.show()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)<br>
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1) <br>
每 5 轮，把学习率乘以 0.1（减小 10 倍）<br>
<br>
for t in range(epochs):<br>
    train_loop(train_loader, model, loss_fn, optimizer)<br>
    test_loop(test_loader, model, loss_fn)<br>
    scheduler.step() # 让减速器起作用<br>

Epoch = 10时的结果：<br>
Epoch 1<br>
Loss: 0.297239  [57600/60000]<br>
Test Error: <br>
 Accuracy: 87.5%, Avg loss: 0.344147 <br>
<br>
Epoch 2<br>
Loss: 0.168551  [57600/60000]<br>
Test Error: <br>
 Accuracy: 87.7%, Avg loss: 0.350386 <br>
<br>
Epoch 3<br>
Loss: 0.131698  [57600/60000]<br>
Test Error: <br>
 Accuracy: 89.0%, Avg loss: 0.323182 <br>
<br>
Epoch 4<br>
Loss: 0.146764  [57600/60000]<br>
Test Error: <br>
 Accuracy: 88.0%, Avg loss: 0.354023 <br>
<br>
Epoch 5<br>
Loss: 0.185207  [57600/60000]<br>
Test Error: <br>
 Accuracy: 88.7%, Avg loss: 0.343801 <br>
<br>
Epoch 6<br>
Loss: 0.099524  [57600/60000]<br>
Test Error: <br>
 Accuracy: 89.9%, Avg loss: 0.313648 <br>
<br>
Epoch 7<br>
Loss: 0.139133  [57600/60000]<br>
Test Error: <br>
 Accuracy: 89.8%, Avg loss: 0.311261 <br>
<br>
Epoch 8<br>
Loss: 0.108912  [57600/60000]<br>
Test Error: <br>
 Accuracy: 89.9%, Avg loss: 0.315970 <br>
<br>
Epoch 9<br>
Loss: 0.062082  [57600/60000]<br>
Test Error: <br>
 Accuracy: 89.8%, Avg loss: 0.320369 <br>
<br>
Epoch 10<br>
Loss: 0.105876  [57600/60000]<br>
Test Error: <br>
 Accuracy: 89.9%, Avg loss: 0.325281 <br>

观察数据它的准确率大致可以达到90%，但也突破不了更高了，测试集的loss大致达到0.32左右<br>

让我们来试试CNN

MLP：把图片拉成一条直线，它会丢失像素之间的位置关系。比如，它不知道左边一个像素和右边一个像素其实是连在一起的一条边。<br>
CNN：它像一个小放大镜（卷积核），在图片上滑来滑去。它能捕捉到局部特征（比如袖口的线条、鞋底的轮廓）。<br>

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class FashionCNN(nn.Module):
    def __init__(self):
        super(FashionCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2) #框出一个2*2的方框，只留下序号最大的
        
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(0.25) # 防止过拟合的秘密武器(丢去25%的神经元，防止特定的神经元进行记忆) 
    
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x))) # 1*28*28➡️32*28*28➡️32*14*14
        x = self.pool(F.relu(self.conv2(x))) # *14*14➡️64*14*14➡️64*7*7
        
        x = x.view(-1, 64 * 7 * 7)
        
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = FashionCNN().to(device)
print(model)

FashionCNN(<br>
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))<br>
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))<br>
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)<br>
  (fc1): Linear(in_features=3136, out_features=128, bias=True)<br>
  (fc2): Linear(in_features=128, out_features=10, bias=True)<br>
  (dropout): Dropout(p=0.25, inplace=False)<br>
)<br>

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_loader, model, loss_fn, optimizer)
    test_loop(test_loader, model, loss_fn)
print("CNN 训练完成！")

loss就放一下最后的来<br>
Epoch 1<br>
Loss: 0.381590  [57600/60000]<br>
Test Error: <br>
 Accuracy: 88.1%, Avg loss: 0.322726 <br>
<br>
Epoch 2<br>
Loss: 0.232530  [57600/60000]<br>
Test Error: <br>
 Accuracy: 89.6%, Avg loss: 0.282284 <br>
<br>
Epoch 3<br>
Loss: 0.347942  [57600/60000]<br>
Test Error: <br>
 Accuracy: 90.6%, Avg loss: 0.257260 <br>
<br>
Epoch 4<br>
Loss: 0.250168  [57600/60000]<br>
Test Error: <br>
 Accuracy: 90.9%, Avg loss: 0.249486 <br>
<br>
Epoch 5<br>
Loss: 0.193854  [57600/60000]<br>
Test Error: <br>
 Accuracy: 91.2%, Avg loss: 0.240125 <br>
<br>
Epoch 6<br>
Loss: 0.030818  [57600/60000]<br>
Test Error: <br>
 Accuracy: 91.3%, Avg loss: 0.243216 <br>
<br>
Epoch 7<br>
Loss: 0.190729  [57600/60000]<br>
Test Error: <br>
 Accuracy: 91.8%, Avg loss: 0.233752 <br>
<br>
Epoch 8<br>
Loss: 0.155328  [57600/60000]<br>
Test Error: <br>
 Accuracy: 91.9%, Avg loss: 0.233524 <br>
<br>
Epoch 9<br>
Loss: 0.278602  [57600/60000]<br>
Test Error: <br>
 Accuracy: 91.8%, Avg loss: 0.242194 <br>
<br>
Epoch 10<br>
Loss: 0.128217  [57600/60000]<br>
Test Error: <br>
 Accuracy: 91.9%, Avg loss: 0.249327 <br>
 <br>
CNN 训练完成！<br>

padding = 1：长宽左右上下各填充1，使其图片与原版一致，可以堆积更多的卷积层，方便做残差连接(一般来说padding = 0)<br>
padding = 0：它的边缘信息只被扫描一次，对于边缘信息不够了解<br>
padding > 1: 使边缘信息更加被注重<br>

最后放一下lr变化的：<br>
Epoch 1<br>
Loss: 0.035143  [57600/60000]<br>
Test Error: <br>
 Accuracy: 92.4%, Avg loss: 0.242689 <br>
<br>
Epoch 2<br>
Loss: 0.094815  [57600/60000]<br>
Test Error: <br>
 Accuracy: 92.4%, Avg loss: 0.250295 <br>
<br>
Epoch 3<br>
Loss: 0.013378  [57600/60000]<br>
Test Error: <br>
 Accuracy: 92.1%, Avg loss: 0.265337 <br>
<br>
Epoch 4<br>
Loss: 0.110929  [57600/60000]<br>
Test Error: <br>
 Accuracy: 92.1%, Avg loss: 0.275465 <br>
<br>
Epoch 5<br>
Loss: 0.279115  [57600/60000]<br>
Test Error: <br>
 Accuracy: 92.1%, Avg loss: 0.279937 <br>
<br>
Epoch 6<br>
Loss: 0.018682  [57600/60000]<br>
Test Error: <br>
 Accuracy: 93.0%, Avg loss: 0.280893 <br>
<br>
Epoch 7<br>
Loss: 0.027328  [57600/60000]<br>
Test Error: <br>
 Accuracy: 93.0%, Avg loss: 0.283847 <br>
<br>
Epoch 8<br>
Loss: 0.011560  [57600/60000]<br>
Test Error: <br>
 Accuracy: 93.0%, Avg loss: 0.287469 <br>
<br>
Epoch 9<br>
Loss: 0.011631  [57600/60000]<br>
Test Error: <br>
 Accuracy: 92.9%, Avg loss: 0.292219 <br>
<br>
Epoch 10<br>
Loss: 0.071546  [57600/60000]<br>
Test Error: <br>
 Accuracy: 93.1%, Avg loss: 0.297039 <br>
<br>
CNN 训练完成！<br>

这个简单的CNN顶多能跑到93.5%，Epoch多跑几轮

升级方法：
1. 加入批归一化 (Batch Normalization)
2. 数据增强 (Data Augmentation)（旋转等方法图片，防止过拟合）
3. 增加模型深度 (Go Deeper) 